## Data paths

In [227]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

PROJECT_ROOT = '../../'

In [228]:
user_path = os.path.join(PROJECT_ROOT, '00_data', '00_raw', 'netflix_users.csv')
watch_history_path = os.path.join(PROJECT_ROOT, '00_data', '00_raw', 'netflix_watch_history.csv')

## User data load

In [229]:
user_df = pd.read_csv(user_path)
user_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9579 entries, 0 to 9578
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   user_id                  9579 non-null   str    
 1   age                      9579 non-null   float64
 2   monthly_spend            9579 non-null   float64
 3   plan_tier                9579 non-null   int64  
 4   is_active                9579 non-null   int64  
 5   email                    9579 non-null   str    
 6   first_name               9579 non-null   str    
 7   last_name                9579 non-null   str    
 8   gender                   8808 non-null   str    
 9   country                  9579 non-null   str    
 10  state_province           9579 non-null   str    
 11  city                     9579 non-null   str    
 12  primary_device           9579 non-null   str    
 13  subscription_start_date  9579 non-null   str    
 14  created_at               9579 non-n

## 사용할 컬럼 선택

In [230]:
print(user_df.columns)
user_columns = ['user_id', 'age', 'plan_tier',
       'subscription_start_date', 'is_active', 'monthly_spend']
raw_df = user_df.copy()
user_df = user_df[user_columns]

Index(['user_id', 'age', 'monthly_spend', 'plan_tier', 'is_active', 'email',
       'first_name', 'last_name', 'gender', 'country', 'state_province',
       'city', 'primary_device', 'subscription_start_date', 'created_at'],
      dtype='str')


## Preprocessing

In [231]:
user_df.loc[(user_df["age"] < 0) | (user_df["age"] > 100), "age"] = None #이상치 없앰
user_df = user_df[~user_df['age'].isna()] #이상치를 포함하여 결측치를 없앰
user_df['age_group'] = user_df['age'] // 10 #구간화
user_df['age_group'] = user_df['age_group'].astype(int)
#user_df = user_df.drop('age', axis=1)
user_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9579 entries, 0 to 9578
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   user_id                  9579 non-null   str    
 1   age                      9579 non-null   float64
 2   plan_tier                9579 non-null   int64  
 3   subscription_start_date  9579 non-null   str    
 4   is_active                9579 non-null   int64  
 5   monthly_spend            9579 non-null   float64
 6   age_group                9579 non-null   int64  
dtypes: float64(2), int64(3), str(2)
memory usage: 524.0 KB


In [232]:
user_df.loc[:,'monthly_spend'] = user_df['monthly_spend'].fillna(user_df['monthly_spend'].median())

In [233]:
user_df["subscription_start_date"] = pd.to_datetime(user_df["subscription_start_date"], errors='coerce')
today = user_df["subscription_start_date"].max()
user_df["subscription_tenure_days"] = (
    today - user_df["subscription_start_date"]
).dt.days
user_df = user_df.drop('subscription_start_date', axis=1)
print(user_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 9579 entries, 0 to 9578
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   9579 non-null   str    
 1   age                       9579 non-null   float64
 2   plan_tier                 9579 non-null   int64  
 3   is_active                 9579 non-null   int64  
 4   monthly_spend             9579 non-null   float64
 5   age_group                 9579 non-null   int64  
 6   subscription_tenure_days  9579 non-null   int64  
dtypes: float64(2), int64(4), str(1)
memory usage: 524.0 KB
None


## watch history load

In [234]:
history_df = pd.read_csv(watch_history_path)
history_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 105000 entries, 0 to 104999
Data columns (total 12 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   session_id              105000 non-null  str    
 1   user_id                 105000 non-null  str    
 2   movie_id                105000 non-null  str    
 3   watch_date              105000 non-null  str    
 4   device_type             105000 non-null  str    
 5   watch_duration_minutes  92668 non-null   float64
 6   progress_percentage     96486 non-null   float64
 7   action                  105000 non-null  str    
 8   quality                 105000 non-null  str    
 9   location_country        105000 non-null  str    
 10  is_download             105000 non-null  bool   
 11  user_rating             21097 non-null   float64
dtypes: bool(1), float64(3), str(8)
memory usage: 8.9 MB


### preprocessing

In [235]:
history_df['watch_date'] = pd.to_datetime(history_df['watch_date'], errors='coerce')
history_df["watch_duration_minutes"] = history_df["watch_duration_minutes"].fillna(0)
history_df["progress_percentage"] = history_df["progress_percentage"].fillna(0)

## Aggregation

In [236]:
watch_count = history_df.groupby('user_id').size().rename('watch_count')
watch_count.value_counts() # minmax의 가치가 있다

watch_count
10    1247
9     1125
11    1117
8     1038
12     981
7      781
13     768
14     643
6      518
15     434
5      323
16     269
17     197
4      176
18     143
19      83
3       56
20      36
21      25
2       23
22       9
23       4
24       2
25       1
26       1
Name: count, dtype: int64

In [237]:
unique_movies = history_df.groupby("user_id")["movie_id"].nunique().rename("unique_movies")
unique_movies.value_counts()

unique_movies
9     1278
10    1270
8     1173
11    1102
7      937
12     919
13     713
6      592
14     507
5      395
15     345
4      208
16     195
17     147
3       66
18      65
19      40
2       25
20      12
22       6
21       5
Name: count, dtype: int64

In [238]:
watch_time = history_df.groupby("user_id")["watch_duration_minutes"].agg(
    total_watch_time="sum",
    avg_watch_time="mean"
)

In [239]:
watch_days = history_df.groupby("user_id")["watch_date"].nunique()
watch_days = watch_days.rename("watch_days")

In [240]:
last_date = history_df['watch_date'].max()

recent = history_df[history_df['watch_date'] >= last_date - pd.Timedelta(days = 31)] #최근 한달 간 활동

recent_watch_count = recent.groupby('user_id').size().rename('recent_watch_count')
recent_watch_count.value_counts()

recent_watch_count
1    2761
2     704
3     128
4      20
5       7
6       2
Name: count, dtype: int64

In [241]:
last_watch = history_df.groupby("user_id")["watch_date"].max()

days_since_last_watch = (
    last_date - last_watch
).dt.days

days_since_last_watch = days_since_last_watch.rename("days_since_last_watch")

In [242]:
avg_progress = history_df.groupby("user_id")["progress_percentage"].mean().rename('avg_progress')
avg_progress.value_counts()

avg_progress
53.500000    12
47.500000     9
38.100000     8
40.000000     8
53.000000     8
             ..
33.454545     1
29.013333     1
56.992308     1
48.512500     1
45.055556     1
Name: count, Length: 8085, dtype: int64

In [243]:
history_df.loc[:,'completed'] = history_df['progress_percentage'] >= 90

completion_rate = history_df.groupby("user_id")["completed"].mean().rename('completion_rate')
completion_rate.value_counts()

completion_rate
0.000000    3946
0.100000     489
0.125000     445
0.111111     441
0.142857     429
            ... 
0.043478       1
0.227273       1
0.545455       1
0.130435       1
0.263158       1
Name: count, Length: 72, dtype: int64

In [244]:
download_ratio = history_df.groupby('user_id')['is_download'].mean().rename('download_ratio')
download_ratio.value_counts()

download_ratio
0.000000    1348
0.250000     616
0.200000     589
0.333333     535
0.166667     489
            ... 
0.529412       1
0.380952       1
0.473684       1
0.727273       1
0.260870       1
Name: count, Length: 94, dtype: int64

In [245]:
avg_rating = history_df.groupby("user_id")["user_rating"].mean().rename('avg_rating')

In [246]:
watch_features = pd.concat([
    watch_count,
    unique_movies,
    watch_time,
    watch_days,
    recent_watch_count,
    days_since_last_watch,
    avg_progress,
    completion_rate,
    download_ratio,
    avg_rating,
], axis=1).reset_index()
watch_features["days_since_last_watch"] = watch_features["days_since_last_watch"].replace(0, 999)

## merge to user dataframe

In [247]:
user_df = user_df.merge(
    watch_features,
    on="user_id",
    how="left"
)
user_df = user_df.fillna(0)
user_df["days_since_last_watch"] = user_df["days_since_last_watch"].replace(0, 999)
user_df

,user_id,age,plan_tier,is_active,monthly_spend,age_group,subscription_tenure_days,watch_count,unique_movies,total_watch_time,avg_watch_time,watch_days,recent_watch_count,days_since_last_watch,avg_progress,completion_rate,download_ratio,avg_rating
0,user_00001,43.0,1,1,36.06,4,480,13,12,643.2,49.476923,12,0.0,62,63.123077,0.307692,0.384615,3.000000
1,user_00002,38.0,4,1,14.59,3,434,15,15,713.4,47.560000,14,0.0,33,49.980000,0.133333,0.266667,3.333333
2,user_00003,32.0,2,0,11.71,3,679,9,8,660.9,73.433333,8,0.0,190,33.644444,0.000000,0.222222,4.000000
3,user_00004,11.0,2,0,28.56,1,345,15,13,1146.7,76.446667,14,1.0,2,41.406667,0.000000,0.200000,4.333333
4,user_00005,21.0,2,0,9.54,2,277,10,9,480.8,48.080000,9,0.0,103,50.890000,0.100000,0.200000,4.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9574,user_09468,37.0,1,0,13.76,3,1039,10,9,847.8,84.780000,9,0.0,81,36.050000,0.000000,0.100000,3.800000
9575,user_09414,39.0,2,0,20.85,3,625,10,10,488.7,48.870000,10,2.0,10,49.050000,0.000000,0.100000,2.333333
9576,user_09414,39.0,2,0,20.85,3,625,10,10,488.7,48.870000,10,2.0,10,49.050000,0.000000,0.100000,2.333333
9577,user_09958,35.0,4,1,8.57,3,270,8,8,540.5,67.562500,8,0.0,39,54.137500,0.125000,0.250000,4.333333


In [248]:
from sklearn.model_selection import train_test_split
train_data, save_data = train_test_split(user_df, test_size = 0.1, stratify = user_df['is_active'], random_state = 42, shuffle = True)

In [249]:
train_path = os.path.join(PROJECT_ROOT, '00_data', '01_interim', 'netflix_users_train.csv')
train_data.to_csv(train_path, index = False)

In [250]:
save_ids = save_data['user_id'].unique()
save_raw_data = raw_df[raw_df['user_id'].isin(save_ids)].copy()

save_path = os.path.join(PROJECT_ROOT, '00_data', '01_interim', 'user_data_for_inference.csv')
save_raw_data.to_csv(save_path, index = False)